In [1]:
import polars as pl
import yaml
from pathlib import Path


with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [29]:
VARIANT = "v2"

In [30]:
stops = pl.DataFrame()
for i in range(256):
    part = pl.read_parquet(
        f"{config['dirs']['output']}/{config['city']}/stops/{VARIANT}/{i:02x}.parquet"
    )
    stops = pl.concat([stops, part])

In [31]:
stops = stops.with_columns(
    pl.col.interval_start.dt.to_string("%Y-%m").alias("year_month")
)

In [32]:
for p in stops.partition_by("year_month"):
    path = f"{config['dirs']['output']}/{config['city']}/monthly_partitioned_stops/{VARIANT}/"
    Path(path).mkdir(exist_ok=True, parents=True)
    p_ = p.sort(["device_id", "interval_start"]).drop("year_month")
    p_.write_parquet(f"{path}/{p['year_month'].first()}.parquet")

    del p_